#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

Supported Languages : Cantonese, Chinese, Dutch, English, French, German, Italian, Japanese, Korean, Polish, Spanish

In [ ]:
# @title ⚙️ 1. Install Dependencies
# @markdown Run this cell to install PyTorch, Torchaudio, Transformers, and other requirements.

import os
from IPython.display import clear_output

print("Installing dependencies. This may take a minute...")
!pip install -q "torch>=2.5.0" "torchaudio>=2.5.0" "transformers>=4.57.0,<5" "soundfile>=0.12" "safetensors>=0.4" "accelerate"

clear_output()
print("✅ Dependencies installed successfully!")

In [ ]:
# @title 🧠 2. Load Audio8-TTS Model
# @markdown Select the model you want to use from the dropdown.

model_choice = "Audio8-TTS-Preview-0.1B (Fast/Hybrid)" # @param ["Audio8-TTS-Preview-0.6B (High Quality)", "Audio8-TTS-Preview-0.1B (Fast/Hybrid)"]

import torch
from transformers import AutoModel, AutoProcessor
from IPython.display import clear_output

# Map the dropdown choice to the correct Hugging Face repository
if "0.6B" in model_choice:
    model_id = "AutoArk-AI/Audio8-TTS-Preview-0.6b"
else:
    model_id = "Audio8/Audio8-TTS-Preview-0.1b"

print(f"Loading {model_id}...")
print("This will download the model weights (might take a few minutes on the first run).")

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

# Load Processor and Model
processor = AutoProcessor.from_pretrained(
    model_id,
    trust_remote_code=True,
)
model = AutoModel.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype=dtype,
).eval().to(device)

clear_output()
print(f"✅ Model {model_choice} loaded successfully on {device.upper()}!")

In [ ]:
# @title 🗣️ 3. Standard Text-to-Speech (No Reference)
# @markdown Enter the text you want the AI to speak. Keep it under 150 characters for the best quality.

target_text = "Hello! This is a test of the Audio8 TTS model running in Google Colab without a reference voice." # @param {type:"string"}

# @markdown ---
# @markdown **Generation Parameters:**
temperature = 0.7 # @param {type:"slider", min:0.1, max:1.5, step:0.1}
top_p = 0.9 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
top_k = 50 # @param {type:"slider", min:1, max:100, step:1}
max_new_tokens = 512 # @param {type:"slider", min:128, max:1024, step:64}

import soundfile as sf
import torch
from IPython.display import Audio, display

print("Generating audio...")

inputs = processor(
    text=[target_text],
    return_tensors="pt",
)
inputs = {name: value.to(device) for name, value in inputs.items()}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)

audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
output_path = "output_no_reference_Audio8-TTS-CoinNoin.wav"
sf.write(output_path, audio, model.config.codec_sample_rate)

print("✅ Generation complete!")
display(Audio(output_path))

In [ ]:
# @title 🎭 4. Zero-Shot Voice Cloning
# @markdown Run this cell. You will immediately be prompted to **upload your reference audio file (.wav)**.
# @markdown After uploading, it will generate the cloned speech based on the text below.

# @markdown ---
# @markdown **Step 1: Text to Generate**
target_text = "This is a demonstration of zero-shot voice cloning. The AI is copying my voice." # @param {type:"string"}

# @markdown **Step 2: Reference Details**
# @markdown Type the *exact* words spoken in the short audio file you are about to upload.
reference_transcript = "Type the exact transcript of the reference recording here." # @param {type:"string"}

# @markdown ---
# @markdown **Generation Parameters:**
temperature = 0.7 # @param {type:"slider", min:0.1, max:1.5, step:0.1}
top_p = 0.9 # @param {type:"slider", min:0.1, max:1.0, step:0.05}
top_k = 50 # @param {type:"slider", min:1, max:100, step:1}

import os
import soundfile as sf
import torch
from google.colab import files
from IPython.display import Audio, display, clear_output

# Trigger runtime file upload
print("📤 Please upload your reference audio file (.wav):")
uploaded = files.upload()

if len(uploaded) == 0:
    print("❌ No file uploaded. Please run the cell again and select a file.")
else:
    reference_audio_path = list(uploaded.keys())[0]
    clear_output()
    print(f"✅ Uploaded {reference_audio_path}. Generating cloned audio...")

    inputs = processor(
        text=[target_text],
        reference_audio=[reference_audio_path],
        reference_text=[reference_transcript],
        return_tensors="pt",
    )
    inputs = {name: value.to(device) for name, value in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            do_sample=True,
            return_dict_in_generate=True,
        )
        waveforms, waveform_lengths = model.decode_audio(output.codes)

    audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
    output_path = "output_cloned_Audio8-TTS-CoinNoin.wav"
    sf.write(output_path, audio, model.config.codec_sample_rate)

    print("✅ Generation complete!")
    display(Audio(output_path))

    # Clean up the uploaded file to keep the workspace tidy
    os.remove(reference_audio_path)